## What is Tool:
外部插件，能夠讓 claude 調用，擁有以下優點:
- **Real-time Information** : Access current data that wasn't available during Claude's training
- **External System Integration** : Connect Claude to databases, APIs, and other services
- **Dynamic Responses** : Provide answers based on the latest available information
- **Structured Interaction** : Claude knows exactly what information it needs and how to ask for it

In [6]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [7]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [8]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

## 1. Tool Function
撰寫 tool function 供 model 呼叫，需要遵守以下 guidelines:
1. **Use descriptive names** : Both your function name and parameter names should clearly indicate their purpose
2. **Validate inputs** : Check that required parameters aren't empty or invalid, and raise errors when they are
3. **Provide meaningful error messages** : Claude can see error messages and might retry the function call with corrected parameters

![](./figure/03/Tool_Function_1.png)

In [9]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

## 2. Tool Schema
將寫好的 tool function 轉換成一種 *類似 JSON 格式* 的 schema，讓 LLM 更好理解 tool 的內容
必備 3 個要素 :
1. **name** - A clear, descriptive name for your tool (like "get_weather")
2. **description** - What the tool does, when to use it, and what it returns
3. **input_schema** - The actual JSON schema describing the function's arguments

其中 Description 對 LLM 的理解至關重要，以下是要遵守的撰寫原則:
- Aim for 3-4 sentences explaining what the tool does
- Describe when Claude should use it
- Explain what kind of data it returns
- Provide detailed descriptions for each argument

![](./figure/03/Tool_Schema_1.png)
![](./figure/03/Tool_Schema_2.png)

### 如何寫一個好的 schema? step 如下:
1. Copy your tool function code
2. Go to Claude and ask it to write a JSON schema for tool calling 
    - 範例: ```"Write a valid JSON schema spec for the purposes of tool calling for this function. Follow the best practices listed in the attached documentation."```
3. Include the [Anthropic documentation on tool use](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview) as context (複製整夜內容當作 attachment 給 claude)
4. Let Claude generate a properly formatted schema following best practices
5. Use the pattern of function_name followed by function_name_schema to keep your schemas organized and easy to match with their corresponding functions.

![](./figure/03/Tool_Schema_3.png)

In [10]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

# Use the pattern of function_name followed by function_name_schema to keep your schemas organized and easy to match with their corresponding functions.

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}

### 為後續做準備，進行型別轉換
import and use the ```ToolParam``` type from the *Anthropic library*

In [11]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

## 3. How to call Claude with schema

### 3-1. Handling message blocks
如何將 tool 進要傳遞給 model 的資訊中?
在 ``` client.messages.create() ``` 中將剛剛撰寫好的 *schema* 放入在 ``` tools ``` 這個 **list** argument 中即可

(這邊故意不使用先前撰寫好的 helper function，單獨處理比較清楚)

In [12]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print(response)

Message(id='msg_011sjqoYFBPFFBCeYmHk6db2', container=None, content=[ToolUseBlock(id='toolu_01KQokFgzjW53pE6yMa71eZR', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=619, output_tokens=63, server_tool_use=None, service_tier='standard'))


在上一格的輸出內容中可以看到模型回傳的資料，content 為主要內容，其中包含了兩種 Block 
1. **"Text" Block** : model 的回覆內容，project 先前的對話回傳 content 裡都只有這種 Block
2. **"Tooluse" Block** :  當 model 要使用 tool 時，會將需求放置此處，並傳遞給 server (在此處就是你手上這台電腦)，裡面包含以下內容:
    - An ID for tracking the tool call
    - The name of the function to call (like "get_current_datetime")
    - Input parameters formatted as a dictionary
    - The type designation "tool_use"

![](./figure/03/Message_Block_1.png)

Claude 並不會儲存對話記憶，因此需要將 model 先前回傳的 content 更新至 conversation history 中

後續在使用時 model 才能夠理解先前的流程，不會亂做

In [13]:
messages.append({
    "role": "assistant",
    "content": response.content
})

print(messages)

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01KQokFgzjW53pE6yMa71eZR', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]


### 使用 tool 流程暫時總結:
1. Send user message with tool schema to Claude
2. Receive assistant message with text block and tool use block
3. Extract tool information and execute the actual function
4. Send tool result back to Claude along with complete conversation history
5. Receive final response from Claude

## 4. 根據 Claude 回傳 content 使用 Tool 
當 server 收到 model 使用 tool 的請求時，會解析 ```ToolUseBlock```，確認模型要如何使用 tool

In [14]:
print(response)

Message(id='msg_011sjqoYFBPFFBCeYmHk6db2', container=None, content=[ToolUseBlock(id='toolu_01KQokFgzjW53pE6yMa71eZR', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=619, output_tokens=63, server_tool_use=None, service_tier='standard'))


In [15]:
response.content[0].input

{'date_format': '%H:%M:%S'}

從上述內容可以獲得 model 要傳入 tool 的 argument，再透過 python 的定位 syntax 讓他自動匹配位置 

In [16]:
get_current_datetime(**response.content[0].input)

'20:34:16'

## 5. 將 Tool result 傳回給 model
將 result 回傳給 model 時需要將資訊放置在 **User message** 中的 **``` Tool Result Block ```** 進行傳輸，告訴 model 執行 tool 發生了什麼
其中會包含以下重要內容:
- **tool_use_id** : Must match the id of the ToolUse block that this ToolResult corresponds to
- **content** : Output from running your tool, serialized as a string
- **is_error** : True if an error occurred

![](./figure/03/Tool_Result_1.png)

以下 code 將 tool result 包裝進 user message 回傳給 claude

包裝後的 ```messages``` 會包含:
- Original user message
- Assistant message with tool use block
- User message with tool result block

In [17]:
# 包裝前
print(messages)

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01KQokFgzjW53pE6yMa71eZR', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]


In [18]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content": "15:04:22",
        "is_error": False
    }]
})

# 包裝後
print(messages)

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01KQokFgzjW53pE6yMa71eZR', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01KQokFgzjW53pE6yMa71eZR', 'content': '15:04:22', 'is_error': False}]}]


接著將 messages 回傳給 claude，其即可依據這些內容給予新的 response

* Note : 要特別說明，即使後續不使用 tool，仍然將 tool schema 附上較好，因為在先前的對話有使用過 tool，還是得讓現在的 model 知道有相對應的知識，不然他沒有記憶，可能會出錯 

![](./figure/03/Tool_Result_2.png)

In [19]:
after_tool_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

In [20]:
print(after_tool_response)

Message(id='msg_013XFJwTozkCCVtjQ9Tr3VuJ', container=None, content=[TextBlock(citations=None, text='The exact time is **15:04:22** (3:04:22 PM).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=699, output_tokens=23, server_tool_use=None, service_tier='standard'))


### 單一 response 中使用多次 tool 該如何回傳? 

claude 可能會在一次 response 中要求使用多次 tool。在每次呼叫 tool 時 claude 都會給予一個 **```id```**，當要將結果回覆給 claude 時務必確保包含 **```id```**，這樣即使在 order 不對的情況下 claude 仍然能夠給予使用者相對應的正確回復 

![](./figure/03/Tool_multiple_Result_1.png)

## 6. The Multi-Turn Tool Pattern
當系統中存在多個 tool 時，claude 可能會在對話流之間交錯使用不同的 tool 以產生結果來回復使用者

例如以下情境:

1. User asks: "What day is 103 days from today?"
2. Claude responds with a tool use block requesting ```get_current_datetime```
3. Your server calls the function and returns the result
4. Claude realizes it needs more information and requests ```add_duration_to_datetime```
5. Your server calls that function and returns the result
6. Claude now has enough information to provide the final answer

![](./figure/03/multi_turn_Tool_1.png)
![](./figure/03/multi_turn_Tool_2.png)

要應付這種情境，需要打造一個 conversation loop，直到 claude 沒有要呼叫 tool 為止 (代表取得階段性結果，準備回覆給 user)

![](./figure/03/multi_turn_Tool_3.png)

### 對 Helper Function 進行重構，後續使用更加方便

These refactoring steps prepare your code for robust tool handling:

- **Flexible message handling** : Your helper functions can now work with different message formats
- **Tool support in chat** : The chat function can receive and pass through tool schemas
- **Full message returns** : You get complete message objects instead of just text, preserving all blocks
- **Text extraction utility** : Easy way to get readable text from complex messages

#### Updating Message Handlers

原先將 message 加入的過程假設只會有 plaintext，若要 general 可以先抓 ```content``` 中的內容出來，後續再進行操作。

In [21]:
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant", 
        "content": message.content if isinstance(message, Message) else message
        }
    messages.append(assistant_message)

### Extracting Text from Messages

僅萃取出 TextBlock，用來將 claude 輸出回覆給 user

In [22]:
def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

### Updating the Chat Function
加入 ```tools``` argument

In [23]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    
    if tools:
        params["tools"] = tools
        
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message

## 7. Implementing Multiple Turns

在 [*6. The Multi-Turn Tool Pattern*]() 提及到在 claude 回傳的 message 不 call tool use 的時候代表這一次的 tool use 告一段落，可以將這一次 response 的 **TextBlock** 打包回傳給 user 作為最終回覆

Claude 的 assistant response 中會有 **`stop_reason`** 這個欄位，這個欄位有以下四種內容:

![Stop Reason](./figure/03/stop_reason.png)

其中當 claude 要 call tool 時，該 *message* 的 `stop reason` 會 set 為 **`tool_use`**，反過來說當對話原先的 stop reason 都是 **`tool_use`**，而某一次 message 突然不是時，代表 claude 已經使用 tool 蒐集完需要的資訊，已經在撰寫給 user 的回覆了，這時就可以把該次 message 的 TextBlock 傳給 user。

<!-- 可以看出 claude 是否有要繼續 call tool -->

```python
if response.stop_reason != "tool_use":
    break  # Claude is done, no more tools needed
```

### Handling Multiple Tool Calls

在過程中還會遇到一個問題，如果 claude 在一個 message 中要使用多次的 tool call，我們必須有系統的記錄每一次 tool 使用情況以及使用結果，主要流程如下圖所示:

![](./figure/03/multi_Tool_call.png)

#### Function Spec

`run_tool` : 
- 負責確認使用哪個 tool，以及 parameter 要帶入什麼
- return 使用結果

`run_tools` : 
- 負責處理該 message 中所有要使用的到的 tool，並將結果以 ToolResultBlock 指定的形式，蒐集打包進 `tool_result_blocks` 中。

    ---

    > `ToolResultBlock` 指定的形式為:
    >```python
    >tool_result_block = {
    >    "type": "tool_result",
    >    "tool_use_id": tool_request.id,
    >    "content": json.dumps(tool_output),
    >    "is_error": False
    >}
    >```
    ---

- 同時使用 `try except` 處理各種形式的 Error，確保打包進 `tool_result_blocks` 的格式是正確的
- return `tool_result_blocks` 




In [24]:
import json


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        # Error Handling
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

### The Conversation Loop

In [25]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        # 當該 response 的 stop_reason 欄位不再是 tool_use 的時候即可將 TextBlock 回傳給 user
        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [26]:
messages = []
add_user_message(
    messages,
    "What is the current time in HH:MM format? Also, what is the current time in SS format?",
)
run_conversation(messages)


The current time is:
- **HH:MM format**: 20:34
- **SS format** (seconds): 18


[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_019GWypcqc5kkpDSKxyqMiEm', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use'),
   ToolUseBlock(id='toolu_01GbFt5UDFCS1YXS8oD72z4W', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_019GWypcqc5kkpDSKxyqMiEm',
    'content': '"20:34"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01GbFt5UDFCS1YXS8oD72z4W',
    'content': '"18"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is:\n- **HH:MM format**: 20:34\n- **SS format** (seconds): 18', type='text')]}]

### Complete Workflow

The complete multi-turn conversation works like this:

1. Send user message to Claude with available tools
2. Claude responds with text and/or tool requests
1. Execute all requested tools and create result blocks
1. Send tool results back as a user message
1. Repeat until Claude provides a final answer

# 8. 需要增加 tool 讓 claude 能夠使用以完成原先目標

## 列出 tool 以及 schema

In [ ]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

pass

## 新增 tool 到對話中讓 claude 知道

In [28]:
import json


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [29]:
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            tools=[
                get_current_datetime_schema,
                add_duration_to_datetime_schema,
                set_reminder_schema,
            ],
        )

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

## 完成後，要求 claude 完成我們提出的任務

In [30]:
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.",
)
run_conversation(messages)

I need to calculate the date that is 177 days after January 1st, 2050, and then set a reminder for you. Let me do that first.
Now I'll set a reminder for your doctor's appointment on June 27, 2050 at 12:00 AM:
----
Setting the following reminder for 2050-06-27T00:00:00:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050 at 12:00 AM**. You'll receive a notification at that time.


[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='I need to calculate the date that is 177 days after January 1st, 2050, and then set a reminder for you. Let me do that first.', type='text'),
   ToolUseBlock(id='toolu_01CKwEeNhRUZZprhoLdMdEXt', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days', 'input_format': '%Y-%m-%d'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01CKwEeNhRUZZprhoLdMdEXt',
    'content': '"Monday, June 27, 2050 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment on June 27, 2050 at 12:00 AM:", type='text'),
   ToolUseBlock(id='toolu_01KH9FdJGVqekam91sBmnr4y', caller=DirectCaller(type=

# **Note**: The Simple Pattern for Adding Tools

Once you have the core tool infrastructure, adding new tools follows this pattern:

1. Create the tool function implementation
1. Define the tool schema
1. Add the schema to the tools list in `run_conversation`
1. Add a case for the tool in `run_tool`

This modular approach makes it easy to expand your AI assistant's capabilities without restructuring existing code. Each new tool integrates seamlessly with the existing conversation flow and tool-handling logic.

# 9. 讓 Claude 能夠平行呼叫 tool : `The Batch Tool`

Claude 能夠在一個 message 中包含很多 tool requset，但實作時通常只會一次呼叫一個，等待回應後下一次 communication 再 call 第二個 tool request。

新增 Batch Tool 後就能讓 claude 在一個 message 中發送多個 tool calling，一次來回即可得到結果 (如果是可以平行處理的 case) 。 不用等待來回通訊，浪費時間以及 token

![](./figure/03/Batch_Tool_1.png)
![](./figure/03/Batch_Tool_2.png)

## Batch Tool 如何運作?

> A batch tool is an additional tool that accepts a list of calls to other tools. Instead of Claude calling multiple tools directly, it calls the batch tool and provides arguments describing which tools it wants to run.

The batch tool receives a list of invocations, where each invocation specifies:

- The `name` of the tool to call
- The `arguments` to pass to that tool

![](./figure/03/Batch_Tool_3.png)

## 1. Implementing the Batch Tool Schema

### define the schema for the batch tool

In [33]:
batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

## 2. Implementing the Batch Function

In [34]:
def run_batch(invocations=[]):
    batch_output = []
    
    for invocation in invocations:
        name = invocation["name"]
        args = json.loads(invocation["arguments"])
        
        tool_output = run_tool(name, args)
        
        batch_output.append({
            "tool_name": name,
            "output": tool_output
        })
    
    return batch_output

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "batch_tool":
        return run_batch(**tool_input)

## 結果:

In [37]:
messages = []
add_user_message(
    messages,
    """Set two reminders for Jan 1st, 2025 at 8 am.:" 
        Happy New Year,
        My dentist appointment
    Use the batch tool to set the alarm in the same message
    """
)
run_conversation(messages)

I'll set two reminders for January 1st, 2025 at 8 AM with the messages you provided.
----
Setting the following reminder for 2025-01-01T08:00:00:
Happy New Year
----
----
Setting the following reminder for 2025-01-01T08:00:00:
My dentist appointment
----
Perfect! I've successfully set both reminders for January 1st, 2025 at 8:00 AM:

1. **Reminder 1**: "Happy New Year"
2. **Reminder 2**: "My dentist appointment"

Both reminders are now scheduled and you'll receive notifications at that time.


[{'role': 'user',
  'content': 'Set two reminders for Jan 1st, 2025 at 8 am.:" \n        Happy New Year,\n        My dentist appointment\n    Use the batch tool to set the alarm in the same message\n    '},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll set two reminders for January 1st, 2025 at 8 AM with the messages you provided.", type='text'),
   ToolUseBlock(id='toolu_01PGhN4VSKKzAjkznTNDr6SB', caller=DirectCaller(type='direct'), input={'content': 'Happy New Year', 'timestamp': '2025-01-01T08:00:00'}, name='set_reminder', type='tool_use'),
   ToolUseBlock(id='toolu_01PZJmVGbrfLnzkEVAikcMrp', caller=DirectCaller(type='direct'), input={'content': 'My dentist appointment', 'timestamp': '2025-01-01T08:00:00'}, name='set_reminder', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01PGhN4VSKKzAjkznTNDr6SB',
    'content': 'null',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01

# 10. Tool Streaming
範例可看 `./source/003_tool_streaming_completed.ipynb`

## Basic Tool Streaming
使用 `streaming` 時需要處理各種 Claude 回傳的 `Event`，例如 **01_Accessing_Claude_with_the_API** 裡處歷過用來傳達 *regular text generating* 的 `ContentBlockDelta`

在 tool 時則變成 `InputJsonEvent`，每個 `InputJsonEvent` 會包含兩個關鍵屬性:
1. `partial_json` - A chunk of JSON representing part of the tool arguments
1. `snapshot` - The cumulative JSON built up from all chunks received so far

會使用以下架構處理 `InputJsonEvent`
```python
    for chunk in stream:
    if chunk.type == "input_json":
        # Process the partial JSON chunk
        print(chunk.partial_json)
        # Or use the complete snapshot so far
        current_args = chunk.snapshot
```

![](./figure/03/tool_streaming_1.png)

## JSON Validation  
### ( Claude 在 return `streaming answer` 之前的細節 )

Claude 並不會在產生 `chunk` 的當下馬上傳回，相反的，他會蒐集並檢查產生的 `chunk` 合不合規則。

首先會等待最上層的 `key-value pairs` 產生完全 (等到四個 `"` 產生後)，與你的 `tool schema` 所期待的內容進行核對，確認無誤後才將目前產生之內容以**原先** `chunk` 的形式回傳

![](./figure/03/tool_streaming_2.png)

舉例，你的 tool 希望收到的格式如下:
```json
{
  "abstract": "This paper presents a novel...",
  "meta": {
    "word_count": 847,
    "review": "This paper introduces QuanNet..."
  }
}
```

The API will:

1. Wait until the entire `abstract` value is complete
1. Validate that `key-value pair` against your schema
1. Send all the buffered chunks for `abstract` at once
1. Repeat the process for the `meta` object

也正是如此，使用 streaming 時，需要等待延遲才能獲得回傳的內容

> The chunks are being held back until a **complete, valid** top-level key-value pair is ready.

![](./figure/03/tool_streaming_3.png)
![](./figure/03/tool_streaming_4.png)

## Fine-Grained Tool Calling
如果需要快速回傳的功能，可以開啟 `Fine-grained tool calling`，能夠讓 API 停止驗證 JSON 完整性，這代表:
- You get chunks as *soon* as Claude generates them
- No buffering delays between top-level keys
- More traditional streaming behavior
- **Critical**: `JSON` validation is disabled - **your code must handle invalid `JSON`**

透過以下方式開啟 
```python
def run_conversation(
    messages, 
    tools=[save_article_schema], 
    fine_grained=True # <- 將這個設為 True
)
```

![](./figure/03/fine-grain_tool_calling_1.png)


## Handling Invalid JSON

When using fine-grained tool calling, Claude might generate invalid `JSON` like `"word_count": undefined` instead of a proper number. Your application needs to handle these cases gracefully:

```python
try:
    parsed_args = json.loads(chunk.snapshot)
except json.JSONDecodeError:
    # Handle invalid JSON appropriately
    print("Received invalid JSON, continuing...")
```

Without fine-grained tool calling, the API's validation would catch this error and potentially wrap problematic values in strings, which might not match your expected schema.

## When to Use Fine-Grained Tool Calling

Consider enabling fine-grained tool calling when:

- You need to show users **real-time progress** on tool argument generation
- You want to start processing partial tool results **as quickly as possible**
- The buffering delays negatively impact your user experience
- You're comfortable implementing robust JSON error handling

# 11. The Text Edit Tool
Claude 內建的 tool.

This tool gives Claude the ability to **work with files and directories just like you would** in a standard text editor.

範例可看 `./source/005_text_editor_tool.ipynb`

> [Text Editor Tool](https://platform.claude.com/docs/en/agents-and-tools/tool-use/text-editor-tool)

The text editor tool provides Claude with a comprehensive set of `file manipulation` capabilities:

- View file or directory contents
- View specific ranges of lines in a file
- Replace text in a file
- Create new files
- Insert text at specific lines in a file
- Undo recent edits to files

![](./figure/03/text_editor_tool_1.png)


## The Implementation Requirements

使用其他 tool 時，我們需要自己撰寫 schema 以及 tool function

在使用 Text Editor Tool 時， Claude 已經知道 schema knowledge，我們只需要撰寫能接收並處理他的 request 的 function (例如: create files, read directories, replace text 等等 )
> 實作 function 在 `./source/005_text_editor_tool.ipynb` 中的 `Implementation of the TextEditorTool` 區塊

### Schema Versions
雖然 Claude 已經有 Text Editor Tool 的 schema，但我們需要指定版本

版本要根據使用的 model 來選擇

```python
def get_text_edit_schema(model):
    if model.startswith("claude-3-7-sonnet"):
        return {
            "type": "text_editor_20250124",
            "name": "str_replace_editor",
        }
    elif model.startswith("claude-3-5-sonnet"):
        return {
            "type": "text_editor_20241022", 
            "name": "str_replace_editor",
        }
```

# 12. Web Search Tool

內建 tool，能夠讓 claude 上網搜尋

只需要提供 schema ，不需要提供實作

範例可參考 `./source/006_web_search_complete.ipynb`

> [使用 web search 時要確定 organization 是否允許](https://console.anthropic.com/settings/privacy)

![](./figure/03/web_search_tool_01.png)

## SetUp Web Search Tool
撰寫包含以下內容的 schema:
```python

web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search", 
    "max_uses": 5
}
```

其中 `max_uses` 限制 claude 能夠做幾次搜尋，以避免過量使用 API

每次搜尋都能獲得多筆結果，但如果需要的話，Claude 可能會要求進行更進一步搜尋

## How the Response Works

When Claude uses the web search tool, the response contains several types of blocks:

- `Text blocks` - Claude's explanation of what it's doing
- `ServerToolUseBlock` - Shows the exact search query Claude used
- `WebSearchToolResultBlock` - Contains the search results
- `WebSearchResultBlock` - Individual search results with titles and URLs
- `Citation blocks` - Text that supports Claude's statements

The response structure lets you see exactly what Claude searched for and which sources it found. 

Citations include the specific text Claude used to support its answers, along with the source URLs.

![](./figure/03/web_search_tool_02.png)
![](./figure/03/web_search_tool_03.png)

## Restricting Search Domains

透過 schema 裡的 `allowed_domains` 欄位，可以限制 web search 到哪幾個 domains 進行搜尋(白名單)

在需要可靠且有權威性的來源時特別有用 (網路上垃圾太多，而且搞不好還是其他 AI 寫出來的)

```python
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}
```

## Rendering Search Results

透過對各種 Block 進行處理，可以客製化後製出想呈現的內容

> The different block types in the response are designed for specific UI rendering:
> - Render text blocks as regular content
> - Display web search results as a list of sources at the top
> - Show citations inline with the text, including the source domain, page title, URL, and quoted text

例如:
![](./figure/03/web_search_tool_04.png)

## Practical Usage

The web search tool works best for:

- Current events and recent developments
- Specialized information not in Claude's training data
- Fact-checking and finding authoritative sources
- Research tasks requiring up-to-date information

Simply include the schema in your tools array when making API calls, and Claude will automatically decide when a web search would help answer the user's question.